## =============================
## ANÁLISE DE COMPORTAMENTO DE PAGAMENTO
## =============================

In [0]:
### Importando bibliotecas
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date
from pyspark.sql.types import *


In [0]:
 #Ler arquivo parquet
df_pagamento_EDA_01 = spark.read.parquet("/Volumes/hackathon_2025/default/source/book_pagamento/dados_pagamento/")

# Visualizar dados
display(df_pagamento_EDA_01)

In [0]:

# PERFILAMENTO AUTOMÁTICO
df_pagamento_EDA_01.summary().display()


In [0]:
# ANÁLISE DE NULOS (QUALIDADE DE DADOS)
from pyspark.sql.functions import col, trim, to_date, desc

total = df_pagamento_EDA_01.count()

nulos_pct = spark.createDataFrame(
    [(c, df_pagamento_EDA_01.filter(col(c).isNull()).count()/total) for c in df_pagamento_EDA_01.columns],
    ["coluna","pct_nulos"]
)

display(nulos_pct.orderBy(desc("pct_nulos")))


In [0]:
from pyspark.sql.functions import avg, count

# PERFIL FINANCEIRO (TICKET MÉDIO)
df_pagamento_EDA_01.groupBy("DW_UN_NEGOCIO") \
  .agg(
      avg("VAL_PAGAMENTO_FATURA").alias("ticket_medio"),
      count("*").alias("qtde_faturas")
  ) \
  .orderBy(desc("ticket_medio")) \
  .display()

In [0]:
# PAGAMENTO POR FORMA E BANCO
df_pagamento_EDA_01.groupBy("DW_FORMA_PAGAMENTO", "DW_BANCO") \
  .agg(
      avg("VAL_PAGAMENTO_FATURA").alias("ticket_medio"),
      count("*").alias("qtde")
  ) \
  .orderBy(desc("ticket_medio")) \
  .display()


In [0]:
#STATUS DE FATURA (RISCO) Taxa de Inadimplência por Tipo
df_pagamento_EDA_01.groupBy("IND_STATUS_FATURA") \
  .count() \
  .withColumn("pct", col("count")/total) \
  .orderBy(desc("count")) \
  .display()


In [0]:
#MULTAS E JUROS (RISCOS FINANCEIROS)
df_pagamento_EDA_01.select(
    avg("VAL_JUROS_MULTAS_ITEM").alias("media_juros"),
    avg("VAL_MULTA_EQUIP_TOTAL").alias("media_multa_equip"),
    avg("VAL_MULTA_FID_ITEM").alias("media_multa_fidelidade")
).display()


In [0]:
#OUTLIERS EM PAGAMENTO (IQR)
from pyspark.sql.functions import col

# Cast VAL_PAGAMENTO_FATURA to double for quantile calculation
df_pagamento_EDA_01_num = df_pagamento_EDA_01.withColumn("VAL_PAGAMENTO_FATURA_NUM", col("VAL_PAGAMENTO_FATURA").cast("double"))


quantis = df_pagamento_EDA_01_num.approxQuantile("VAL_PAGAMENTO_FATURA_NUM", [0.25, 0.75], 0.01)
q1, q3 = quantis
iqr = q3 - q1
lim_inf = q1 - 1.5*iqr
lim_sup = q3 + 1.5*iqr

df_pagamento_EDA_01_num.filter(
    (col("VAL_PAGAMENTO_FATURA_NUM") < lim_inf) |
    (col("VAL_PAGAMENTO_FATURA_NUM") > lim_sup)
).select("NUM_CPF", "VAL_PAGAMENTO_FATURA").display()

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

# DISTRIBUIÇÃO DE PAGAMENTOS
pdf = df_pagamento_EDA_01.select("VAL_PAGAMENTO_FATURA").sample(0.1, seed=42).toPandas()

plt.figure(figsize=(6,4))
sns.histplot(pdf["VAL_PAGAMENTO_FATURA"].dropna(), kde=True)
plt.title("Distribuição do Valor Pago por Fatura")
plt.xlabel("VAL_PAGAMENTO_FATURA")
plt.ylabel("Frequência")
plt.tight_layout()
plt.show()

In [0]:
# CORRELAÇÃO ENTRE VALORES FINANCEIROS
pdf_corr = df_pagamento_EDA_01.select(
    "VAL_PAGAMENTO_FATURA",
    "VAL_DESCONTO_ITEM",
    "VAL_JUROS_MULTAS_ITEM",
    "VAL_MULTA_EQUIP_TOTAL"
).sample(0.1, seed=42).toPandas()

corr = pdf_corr.corr()

plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True)
plt.title("Correlação Financeira")
plt.show()


In [0]:
# CARDINALIDADE (QUANTOS CLIENTES ÚNICOS)
df_pagamento_EDA_01.select("NUM_CPF").distinct().count()


In [0]:
# PERFIL POR ÁREA E NEGÓCIO
df_pagamento_EDA_01.groupBy("DW_AREA","DW_UN_NEGOCIO") \
  .agg(
      avg("VAL_PAGAMENTO_FATURA").alias("ticket_medio"),
      count("*").alias("qtde_faturas")
  ) \
  .orderBy(desc("ticket_medio")) \
  .display()


In [0]:

    from pyspark.sql.functions import count, countDistinct, sum, avg, round, col, desc

# Analisa comportamento de pagamento dos clientes

# 4.1 Status das faturas
# print("\n--- Distribuição por Status da Fatura ---")
df_pagamento_EDA_01.groupBy("IND_STATUS_FATURA") \
    .agg(
        count("*").alias("Qtd_Faturas"),
        countDistinct("DW_NUM_CLIENTE").alias("Qtd_Clientes"),
        sum("VAL_PAGAMENTO_FATURA").alias("Valor_Total"),
        avg("VAL_PAGAMENTO_FATURA").alias("Ticket_Medio")
    ) \
    .withColumn("Pct", round(col("Qtd_Faturas") / df_pagamento_EDA_01.count() * 100, 2)) \
    .orderBy(desc("Qtd_Faturas")) \
    .show(truncate=False)

In [0]:
 # 4.2 Análise de inadimplência
   # 4.2 Análise de inadimplência
#print("\n--- Taxa de Inadimplência por Tipo de Fatura ---")
df_pagamento_EDA_01.groupBy("DW_TIPO_FATURA", "IND_STATUS_FATURA") \
    .agg(count("*").alias("Qtd")) \
    .groupBy("DW_TIPO_FATURA") \
    .pivot("IND_STATUS_FATURA") \
    .sum("Qtd") \
    .na.fill(0) \
    .show(truncate=False)

# =================================
##  ANÁLISE TEMPORAL
# =================================

In [0]:

#Análise por área geográfica e unidade de negócio
df_pagamento_EDA_01.groupBy("DW_AREA", "DW_TIPO_FATURA") \
        .agg(
            countDistinct("DW_NUM_CLIENTE").alias("Qtd_Clientes"),
            count("*").alias("Qtd_Faturas"),
            sum("VAL_PAGAMENTO_FATURA").alias("Receita_Total"),
            avg("VAL_PAGAMENTO_FATURA").alias("Ticket_Medio")
        ) \
        .orderBy(desc("Receita_Total")) \
        .show(30, truncate=False)

In [0]:
df_pagamento_EDA_01.groupBy("DW_UN_NEGOCIO", "DW_TIPO_FATURA") \
        .agg(
            countDistinct("DW_NUM_CLIENTE").alias("Qtd_Clientes"),
            count("*").alias("Qtd_Faturas"),
            sum("VAL_PAGAMENTO_FATURA").alias("Receita_Total"),
            avg("VAL_PAGAMENTO_FATURA").alias("Ticket_Medio")
        ) \
        .orderBy(desc("Qtd_Clientes")) \
        .show(30, truncate=False)
    
    #return df_pagamento_EDA_01

In [0]:
# 1. Taxa de Inadimplência por Tipo de Fatura
# Considerando que 'Paga' é o status de sucesso

from pyspark.sql.functions import when, avg

df_pagamento_EDA_01_inad = df_pagamento_EDA_01.withColumn(
    "is_atrasada",
    when(col("IND_STATUS_FATURA").isin(["VENCIDA", "INADIMPLENTE"]), 1).otherwise(0)
)

taxa_tipo = df_pagamento_EDA_01_inad.groupBy("DW_TIPO_FATURA") \
    .agg((avg("is_atrasada") * 100).alias("Taxa %")) \
    .orderBy(desc("Taxa %"))

display(taxa_tipo)

In [0]:
# Indicador de inadimplência (binário)

import numpy as np

df['FLG_INADIMPLENTE'] = np.where(
    df_pagamento_EDA_01['IND_STATUS_FATURA'].isin(['ATRASADA','PENDENTE','INADIMPLENTE']), 1, 0
)

df['VAL_JUROS_MULTAS_TOTAL'] = (
    df['VAL_JUROS_MULTAS_ITEM']
  + df['VAL_MULTA_EQUIP_TOTAL']
  + df['VAL_MULTA_FID_ITEM']
)


In [0]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(df_pagamento_EDA_01['FLG_INADIMPLENTE'], bins=10)
plt.title('Distribuição da Taxa de Inadimplência por Tipo de Fatura')
plt.xlabel('Taxa de Inadimplência')
plt.ylabel('Frequência')
plt.show()


In [0]:

import matplotlib.pyplot as plt

plt.figure()
df_pagamento_EDA_01['IND_STATUS_FATURA'].value_counts().plot(kind='bar')
plt.title('Distribuição de Status das Faturas')
plt.xlabel('Status')
plt.ylabel('Quantidade')
plt.show()

In [0]:
df_pagamento_EDA_01['DW_FORMA_PAGAMENTO'] = np.where(
    df_pagamento_EDA_01['DW_FORMA_PAGAMENTO'].str.contains('CART', case=False, na=False) |
    df_pagamento_EDA_01['DW_TIPO_PAGAMENTO'].str.contains('CART', case=False, na=False),
    'CARTAO_CREDITO',
    np.where(
        df_pagamento_EDA_01['DW_FORMA_PAGAMENTO'].str.contains('BOLE', case=False, na['COD_BARRAS'].notna(), 
        'BOLETO', 
        'OUTROS'
    )
)